# Incremental Load: Silver tables
**Source:** `bronze` | **Target:** `silver`

## Table: silver_crm_sales_details

In [ ]:
-- cte to add order_date (watermark column) in bronze table (sales_details)
WITH bronze_with_date AS (
    SELECT *,
    TO_DATE(sls_order_dt, 'yyyyMMdd') AS order_date
    FROM sales_lakehouse.dbo.bronze_crm_sales_details
),
-- cte for watermark
watermark AS (
SELECT
last_loaded_value
FROM sales_lakehouse.dbo.incremental_load_control
WHERE table_name = 'crm_sales_details'
)

-- insering the latest data into silver table
INSERT INTO sales_lakehouse.dbo.silver_crm_sales_details

SELECT 
    sls_ord_num,
    sls_prd_key,
    CAST(sls_cust_id AS INT) AS sls_cust_id,

    -- Data handling
    CASE WHEN sls_order_dt = 0 OR LEN(sls_order_dt) !=8 THEN NULL
         ELSE TO_DATE(CAST(sls_order_dt AS STRING), 'yyyyMMdd')
    END AS sls_order_dt,

-- applying same conditions for future proof
    CASE WHEN sls_ship_dt = 0 OR LEN(sls_ship_dt) !=8 THEN NULL
         ELSE TO_DATE(CAST(sls_ship_dt AS STRING), 'yyyyMMdd')
    END AS sls_ship_dt,

-- applying same conditions for future proof
    CASE WHEN sls_due_dt = 0 OR LEN(sls_due_dt) !=8 THEN NULL
         ELSE TO_DATE(TRIM(CAST(sls_due_dt AS STRING)), 'yyyyMMdd')
    END AS sls_due_dt,

-- applying business rules for sales, quantity and price
    CAST(
        CASE WHEN sls_sales <= 0 OR sls_sales IS NULL OR sls_sales != sls_quantity * ABS(sls_price)
            THEN sls_quantity * ABS(sls_price)
            ELSE sls_sales
        END 
    AS INT) AS sls_sales,

    CAST(sls_quantity AS INT) AS sls_quantity,

    CAST(
        CASE WHEN sls_price <= 0 OR sls_price IS NULL
            THEN sls_sales / NULLIF(sls_quantity,0)       -- handling divide by 0 error
            ELSE sls_price
        END
    AS INT) AS sls_price,

-- adding metadata columns
    CURRENT_TIMESTAMP() AS meta_load_timestamp,
    'CRM' AS meta_source_system,
    'crm_sales_details' AS meta_table_name
 FROM bronze_with_date b
 CROSS JOIN watermark w
 WHERE 
       b.order_date > w.last_loaded_value   -- loading only the new data
       OR (
       b.order_date IS NULL                -- keeping the NULL order_dates as well
       AND NOT EXISTS(                     -- this helps to avoid loading the NULL order_dates as ord_num will always be unique
           SELECT 1
           FROM sales_lakehouse.dbo.bronze_crm_sales_details s
           WHERE s.sls_ord_num = b.sls_ord_num
           )
        );

### Updating the watermark / Incremental Load Control table after first load/insert

In [ ]:
-- Updating the watermark table after first load/insert
UPDATE sales_lakehouse.dbo.incremental_load_control
SET last_loaded_value = COALESCE(
    (SELECT MAX(sls_order_dt) 
    FROM sales_lakehouse.dbo.silver_crm_sales_details),
    last_loaded_value
)
WHERE table_name = 'crm_sales_details';